# Sesión 7: Consultas a Tablas Relacionadas en SQL
## Clase 1 - Consultas Avanzadas y Relaciones

En esta sesión aprenderemos:
- Modelos de datos (conceptual, lógico, físico)
- Integridad referencial y sus reglas
- Claves primarias y foráneas
- Relaciones 1:N, N:1, N:M
- JOINs para consultar tablas relacionadas

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("✅ Conexión SQLite establecida para Sesión 7")

## SETUP: Crear modelo relacional de distribución de alimentos (Slide 31-33)

In [ ]:
# Slide 33: Crear tablas con relaciones

# 1. Tabla CIUDADES (base)
cursor.execute('''
    CREATE TABLE ciudades (
        id_ciudad INTEGER PRIMARY KEY,
        nombre_ciudad TEXT NOT NULL
    )
''')

# 2. Tabla CLIENTES con FK a ciudades
cursor.execute('''
    CREATE TABLE clientes (
        id_cliente INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        ciudad_id INTEGER,
        FOREIGN KEY (ciudad_id) REFERENCES ciudades(id_ciudad)
    )
''')

# 3. Tabla PRODUCTOS
cursor.execute('''
    CREATE TABLE productos (
        id_producto INTEGER PRIMARY KEY,
        nombre_producto TEXT NOT NULL,
        precio NUMERIC CHECK (precio > 0)
    )
''')

# 4. Tabla PEDIDOS con FK a clientes
cursor.execute('''
    CREATE TABLE pedidos (
        id_pedido INTEGER PRIMARY KEY,
        id_cliente INTEGER NOT NULL,
        fecha DATE NOT NULL,
        FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente)
    )
''')

# 5. Tabla DETALLE_PEDIDO (relación N:M entre pedidos y productos)
cursor.execute('''
    CREATE TABLE detalle_pedido (
        id_detalle INTEGER PRIMARY KEY,
        id_pedido INTEGER NOT NULL,
        id_producto INTEGER NOT NULL,
        cantidad INTEGER CHECK (cantidad > 0),
        FOREIGN KEY (id_pedido) REFERENCES pedidos(id_pedido),
        FOREIGN KEY (id_producto) REFERENCES productos(id_producto)
    )
''')

print("✅ Tablas creadas con relaciones:")
print("   • ciudades (base)")
print("   • clientes (FK → ciudades)")
print("   • productos (independiente)")
print("   • pedidos (FK → clientes) - Relación 1:N")
print("   • detalle_pedido (FK → pedidos, FK → productos) - Relación N:M")

### Insertar datos de ejemplo

In [ ]:
# Insertar ciudades
ciudades = [
    (1, 'Santiago'),
    (2, 'Valparaíso'),
    (3, 'Concepción'),
    (4, 'Arica')
]
cursor.executemany('INSERT INTO ciudades VALUES (?, ?)', ciudades)

# Insertar clientes
clientes = [
    (1, 'Camila Rojas', 2),
    (2, 'Luis Pérez', 1),
    (3, 'Daniela Soto', 3),
    (4, 'Roberto García', 2),
    (5, 'Ana Martínez', 1),
    (6, 'Jorge López', 4)
]
cursor.executemany('INSERT INTO clientes VALUES (?, ?, ?)', clientes)

# Insertar productos
productos = [
    (1, 'Leche Descremada', 1200),
    (2, 'Pan Integral', 1500),
    (3, 'Queso Fresco', 2500),
    (4, 'Yogur Natural', 800),
    (5, 'Mantequilla', 3000),
    (6, 'Huevos (Docena)', 1800)
]
cursor.executemany('INSERT INTO productos VALUES (?, ?, ?)', productos)

# Insertar pedidos
pedidos = [
    (1, 1, '2024-06-01'),
    (2, 2, '2024-06-05'),
    (3, 3, '2024-06-10'),
    (4, 1, '2024-06-15'),
    (5, 4, '2024-07-01'),
    (6, 2, '2024-07-05'),
    (7, 5, '2024-07-10'),
    (8, 3, '2024-07-15')
]
cursor.executemany('INSERT INTO pedidos VALUES (?, ?, ?)', pedidos)

# Insertar detalle de pedidos (N:M)
detalle = [
    (1, 1, 1, 2),   # Pedido 1: 2 leches
    (2, 1, 3, 1),   # Pedido 1: 1 queso
    (3, 2, 2, 3),   # Pedido 2: 3 panes
    (4, 2, 4, 2),   # Pedido 2: 2 yogures
    (5, 3, 1, 1),   # Pedido 3: 1 leche
    (6, 3, 5, 2),   # Pedido 3: 2 mantequillas
    (7, 4, 2, 5),   # Pedido 4: 5 panes
    (8, 4, 6, 1),   # Pedido 4: 1 docena huevos
    (9, 5, 1, 3),   # Pedido 5: 3 leches
    (10, 6, 3, 2),  # Pedido 6: 2 quesos
    (11, 7, 4, 4),  # Pedido 7: 4 yogures
    (12, 8, 5, 1)   # Pedido 8: 1 mantequilla
]
cursor.executemany('INSERT INTO detalle_pedido VALUES (?, ?, ?, ?)', detalle)
conn.commit()

print(f"✅ Datos insertados:")
print(f"   • {len(ciudades)} ciudades")
print(f"   • {len(clientes)} clientes")
print(f"   • {len(productos)} productos")
print(f"   • {len(pedidos)} pedidos")
print(f"   • {len(detalle)} items en pedidos")

## 1. MODELOS DE DATOS (Slides 6-12)

In [ ]:
# Slide 7-8: Componentes y tipos de modelos
print("SLIDE 7-8: Componentes del Modelo Relacional")
print("="*70)

componentes = pd.DataFrame({
    'Componente': ['Entidades', 'Atributos', 'Relaciones', 'Reglas de Negocio'],
    'Descripción': [
        'Objetos del mundo real (Cliente, Pedido, Producto)',
        'Propiedades de entidades (nombre, precio, fecha)',
        'Vínculos entre entidades (1:N, N:M)',
        'Restricciones (pedido requiere cliente, cantidad > 0)'
    ],
    'Ejemplo en nuestro modelo': [
        'clientes, pedidos, productos',
        'c.nombre, p.fecha, pr.precio',
        'Cliente → Pedidos (1:N), Pedido → Productos (N:M)',
        'FK, CHECK, NOT NULL'
    ]
})

print(componentes.to_string(index=False))

In [ ]:
# Slide 8-9: Tipos de modelos
types_modelos = pd.DataFrame({
    'Tipo': ['Conceptual', 'Lógico', 'Físico'],
    'Abstracción': ['Alto', 'Medio', 'Bajo'],
    'Detalle técnico': ['Bajo', 'Medio-Alto', 'Muy alto'],
    'Audiencia': ['Usuarios negocio', 'Analistas/Diseñadores', 'Admins BD'],
    'Ejemplo': [
        'DER (Diagrama Entidad-Relación)',
        'Tablas, columnas, FK, PK',
        'Índices, particiones, almacenamiento'
    ]
})

print("\nSLIDE 8-9: Tipos de Modelos de Datos")
print("\n" + types_modelos.to_string(index=False))

## 2. INTEGRIDAD REFERENCIAL (Slides 13-19)

In [ ]:
print("SLIDE 14: Principios de Integridad Referencial")
print("="*70)
print("""
Principio central:
"Toda FK debe tener un valor coincidente en la PK de la tabla referenciada,
o ser nula si la relación lo permite."

En nuestro modelo:
• Cada pedido.id_cliente DEBE existir en clientes.id_cliente
• Cada pedido.id_producto DEBE existir en productos.id_producto
• Cada cliente.ciudad_id DEBE existir en ciudades.id_ciudad

Beneficio: Evita registros "huérfanos" (sin relación válida)
""")

In [ ]:
# Slide 15-16: ON DELETE y ON UPDATE
print("SLIDE 15-16: Cláusulas ON DELETE y ON UPDATE")
print("="*70)

clausulas = pd.DataFrame({
    'Cláusula': ['ON DELETE CASCADE', 'ON DELETE SET NULL', 'ON DELETE RESTRICT', 'ON UPDATE CASCADE'],
    'Comportamiento': [
        'Elimina registros dependientes automáticamente',
        'Establece FK en NULL al eliminar padre',
        'Impide eliminar si hay dependencias (por defecto)',
        'Propaga cambios de PK a FKs relacionadas'
    ],
    'Ejemplo práctico': [
        'Eliminar cliente → Elimina todos sus pedidos',
        'Eliminar ciudad → FK ciudad_id en clientes = NULL',
        'No permite eliminar cliente con pedidos',
        'Cambiar id_cliente → Actualiza FK en pedidos'
    ]
})

print("\n" + clausulas.to_string(index=False))

In [ ]:
# Slide 17: Validar integridad referencial
print("\nSLIDE 17: Métrica - Contar Claves Huérfanas")
print("="*70)

df_huerfanas = pd.read_sql_query(
    """SELECT COUNT(*)
    FROM detalle_pedido dp
    WHERE NOT EXISTS (
        SELECT 1 FROM pedidos p WHERE p.id_pedido = dp.id_pedido
    )""",
    conn
)

print(f"\nClaves huérfanas en detalle_pedido: {df_huerfanas.iloc[0, 0]}")
print("✅ Integridad referencial validada: Sin huérfanos")

## 3. CLAVES PRIMARIAS Y FORÁNEAS (Slides 20-27)

In [ ]:
print("SLIDE 21-22: Definición y Tipos de Claves")
print("="*70)

print("""
Clave Primaria (PK):
• Identifica ÚNICAMENTE cada fila
• NO puede ser nula
• NO puede cambiar frecuentemente
• Ejemplo en nuestro modelo: id_cliente, id_pedido

Clave Foránea (FK):
• Referencia PK de otra tabla
• Establece relaciones
• Puede ser nula si la relación es opcional
• Ejemplo: pedidos.id_cliente → clientes.id_cliente

Claves Compuestas:
• Dos o más columnas combinadas forman la PK
• Ejemplo: detalle_pedido.id_pedido + id_producto
""")

## 4. RELACIONES 1:N, N:1, N:M (Slide 23-24)

In [ ]:
print("SLIDE 23: Tipos de Relaciones")
print("="*70)

relaciones = pd.DataFrame({
    'Tipo': ['Uno a Muchos (1:N)', 'Muchos a Uno (N:1)', 'Muchos a Muchos (N:M)'],
    'Descripción': [
        'Un registro A se relaciona con múltiples en B',
        'Múltiples registros de A apuntan a uno de B',
        'Se necesita tabla intermedia'
    ],
    'Ejemplo': [
        'Clientes → Pedidos: 1 cliente, muchos pedidos',
        'Pedidos → Clientes: muchos pedidos, 1 cliente',
        'Pedidos ↔ Productos vía detalle_pedido'
    ],
    'Implementación': [
        'FK en tabla muchos',
        'FK en tabla muchos',
        'Tabla intermedia con 2 FK'
    ]
})

print("\n" + relaciones.to_string(index=False))

## 5. CONSULTAS CON JOINs (Slide 34)

In [ ]:
# Slide 34: Consulta guiada con múltiples JOINs
print("SLIDE 34: Consulta Guiada - Cliente, Ciudad y Total Productos")
print("="*70)

df_query = pd.read_sql_query(
    """SELECT 
        c.nombre AS cliente, 
        ci.nombre_ciudad AS ciudad, 
        SUM(dp.cantidad) AS total_productos_comprados,
        COUNT(DISTINCT p.id_pedido) AS numero_pedidos
    FROM clientes c
    JOIN ciudades ci ON c.ciudad_id = ci.id_ciudad
    LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
    LEFT JOIN detalle_pedido dp ON p.id_pedido = dp.id_pedido
    GROUP BY c.id_cliente, c.nombre, ci.nombre_ciudad
    ORDER BY total_productos_comprados DESC""",
    conn
)

print("\nResultado:")
print(df_query.to_string(index=False))
print("\n✅ JOINs utilizados:")
print("   • JOIN clientes-ciudades (1:N) - INNER")
print("   • LEFT JOIN clientes-pedidos (1:N)")
print("   • LEFT JOIN pedidos-detalle (1:N)")
print("   • GROUP BY para agregación")

## 6. VALIDAR INTEGRIDAD (Slide 25)

In [ ]:
print("SLIDE 25: Validación de Integridad Cruzada")
print("="*70)

# Verificar que NO hay pedidos con clientes inválidos
df_pedidos_invalidos = pd.read_sql_query(
    """SELECT COUNT(*)
    FROM pedidos p
    WHERE NOT EXISTS (
        SELECT 1 FROM clientes c WHERE c.id_cliente = p.id_cliente
    )""",
    conn
)

# Verificar que NO hay detalles con pedidos inválidos
df_detalles_invalidos = pd.read_sql_query(
    """SELECT COUNT(*)
    FROM detalle_pedido dp
    WHERE NOT EXISTS (
        SELECT 1 FROM pedidos p WHERE p.id_pedido = dp.id_pedido
    )""",
    conn
)

print(f"\n✅ Validaciones:")
print(f"   Pedidos con cliente inválido: {df_pedidos_invalidos.iloc[0, 0]}")
print(f"   Detalles con pedido inválido: {df_detalles_invalidos.iloc[0, 0]}")
print(f"\n✅ RESULTADO: Integridad referencial VÁLIDA en toda la BD")

## 7. ANÁLISIS DE RELACIONES (Slide 32-34)

In [ ]:
print("SLIDE 32-34: Interpretación de Estructura Relacional")
print("="*70)

print("""
DIAGRAMA CONCEPTUAL:

    ┌─────────────┐
    │  CIUDADES   │
    ├─────────────┤
    │ id_ciudad   │ (PK)
    └─────────────┘
            ▲
            │ (1:N)
            │ ciudad_id (FK)
    ┌──────────────────┐
    │    CLIENTES      │
    ├──────────────────┤
    │ id_cliente (PK)  │
    │ nombre           │
    │ ciudad_id (FK)   │
    └──────────────────┘
            │
            │ (1:N)
            │ id_cliente (FK)
    ┌────────────────────┐
    │     PEDIDOS        │
    ├────────────────────┤
    │ id_pedido (PK)     │
    │ id_cliente (FK)    │ ──► Relación N:1 con CLIENTES
    │ fecha              │
    └────────────────────┘
            │
            │ (1:N)
            │ id_pedido (FK)
    ┌─────────────────────────────┐
    │   DETALLE_PEDIDO (N:M)      │
    ├─────────────────────────────┤
    │ id_pedido (FK) ─────┐       │
    │ id_producto (FK) ────► Tabla puente
    │ cantidad            │
    └─────────────────────────────┘
                            │
                            │ (1:N inversa)
                            │ id_producto (FK)
                    ┌─────────────────┐
                    │   PRODUCTOS     │
                    ├─────────────────┤
                    │ id_producto (PK)│
                    │ nombre_producto │
                    │ precio          │
                    └─────────────────┘

ENTIDADES: 5 (ciudades, clientes, pedidos, detalle_pedido, productos)
REST RIMCCIONES:
• Ciudades: Base (sin dependencias)
• Clientes: dependen de Ciudades (1:N)
• Pedidos: dependen de Clientes (1:N)
• Detalle: puente entre Pedidos y Productos (N:M)
""")

print("\nVALIDACIÓN DE ESTRUCTURA:")
print(f"✅ PK definidas en todas las tablas")
print(f"✅ FK correctamente declaradas")
print(f"✅ Relaciones respetan normalización (3FN)")
print(f"✅ Integridad referencial garantizada")

## 8. EJEMPLOS DE JOINs COMUNES

In [ ]:
# INNER JOIN: Solo registros coincidentes
print("INNER JOIN: Solo clientes con pedidos")
print("="*70)

df_inner = pd.read_sql_query(
    """SELECT c.nombre, COUNT(p.id_pedido) as num_pedidos
    FROM clientes c
    INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
    GROUP BY c.id_cliente, c.nombre""",
    conn
)

print(f"\nResultado: {len(df_inner)} clientes con pedidos")
print(df_inner.to_string(index=False))

In [ ]:
# LEFT JOIN: Todos clientes, aunque no tengan pedidos
print("\nLEFT JOIN: Todos los clientes (con o sin pedidos)")
print("="*70)

df_left = pd.read_sql_query(
    """SELECT c.nombre, COUNT(p.id_pedido) as num_pedidos
    FROM clientes c
    LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
    GROUP BY c.id_cliente, c.nombre
    ORDER BY num_pedidos DESC""",
    conn
)

print(f"\nResultado: {len(df_left)} clientes (incluye sin pedidos)")
print(df_left.to_string(index=False))

## RESUMEN (Slide 36)

In [ ]:
resumen = """
    SLIDE 36: RESUMEN DE SESIÓN 7
    ═════════════════════════════════════════════════════════════
    
    1️⃣  MODELOS DE DATOS
        Conceptual → Lógico → Físico
        • Entidades, Atributos, Relaciones, Reglas
    
    2️⃣  INTEGRIDAD REFERENCIAL
        FK debe coincidir con PK o ser nula
        • ON DELETE CASCADE/SET NULL/RESTRICT
        • Validación de claves huérfanas
    
    3️⃣  CLAVES PRIMARIAS Y FORÁNEAS
        PK: Identifica únicamente
        FK: Establece relaciones
        • Simples o compuestas
    
    4️⃣  RELACIONES
        1:N → FK en tabla muchos
        N:M → Tabla intermedia con 2 FK
        • Ejemplo: Clientes-Pedidos-Productos
    
    5️⃣  JOINs
        INNER JOIN: Solo coincidencias
        LEFT JOIN: Todos de tabla izquierda
        • Múltiples JOINs para análisis complejos
    
    ═════════════════════════════════════════════════════════════
    PRÓXIMA SESIÓN:
    
    Operaciones INNER JOIN y LEFT JOIN
    • Combinaciones más complejas
    • Subconsultas
    • Análisis avanzados
"""

print(resumen)